In [2]:
import os
import glob
import scanpy as sc
import pandas as pd
import scanpy as sc
import numpy as np

# Get the current working directory
cwd = os.getcwd()

# Define input and output directories
input_dir = os.path.join(cwd, "Individual-h5ad")
output_dir = os.path.join(cwd, "Individual-QC-h5ad5")

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Find all H5 files in the specified input directory
h5ad_files = glob.glob(os.path.join(input_dir, "TSP27_Trachae*.h5ad"))


for h5ad_file in h5ad_files:
    print(f"Processing {h5ad_file}...")

    # Load the data using Scanpy's function for 10X Genomics h5 files
    adata = sc.read_h5ad(h5ad_file)

    # Annotate mitochondrial, ribosomal, and hemoglobin genes
    adata.var["mt"] = adata.var_names.str.startswith("MT-")  # Mitochondrial genes
    adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))  # Ribosomal genes
    adata.var["hb"] = adata.var_names.str.contains("^HB[^(P)]")  # Hemoglobin genes

    # Compute quality control metrics and log-transform counts
    sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True, log1p=True)
   
    # Generate QC plots to visualize quality metrics
    sc.pl.violin(adata, ["n_genes_by_counts", "total_counts", "pct_counts_mt"], jitter=0.4, multi_panel=True)
    sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

    # Filter out low-quality cells and genes
    sc.pp.filter_cells(adata, min_genes=100)  # Remove cells with fewer than 100 detected genes and less tha 200 counts
    sc.pp.filter_genes(adata, min_cells=3)  # Remove genes detected in fewer than 3 cells

    # Apply Scrublet to detect and annotate doublets in the dataset
    sc.pp.scrublet(adata)

    # Copy decontXcounts to X to overide the normalization
    adata.X = adata.layers["decontXcounts"].copy()

    # Normalize total counts per cell to median total counts
    sc.pp.normalize_total(adata)

    # Log-transform the normalized data to stabilize variance
    sc.pp.log1p(adata)

    # Identify and annotate highly variable genes (HVGs)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    sc.pl.highly_variable_genes(adata)

    # Perform principal component analysis (PCA) for dimensionality reduction
    sc.tl.pca(adata)
    sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)
    sc.pl.pca(adata, color="total_counts")

    # Compute nearest-neighbor graph for downstream clustering and visualization
    sc.pp.neighbors(adata)

    # Compute and visualize UMAP embeddings
    sc.tl.umap(adata)
    sc.pl.umap(adata, size=2, color="total_counts")
    sc.tl.leiden(adata, flavor="igraph", n_iterations=2)
    sc.pl.umap(adata, color=["leiden", "predicted_doublet", "doublet_score"], wspace=0.5, size=3)
    sc.pl.umap(adata, color=["leiden", "log1p_total_counts", "pct_counts_mt", "log1p_n_genes_by_counts"], wspace=0.5, ncols=2)

    # Perform Leiden clustering at multiple resolutions
    for res in [0.02, 0.5, 2.0]:
        sc.tl.leiden(adata, key_added=f"leiden_res_{res:4.2f}", resolution=res, flavor="igraph")

    # Visualize clustering results on UMAP at different resolutions
    sc.pl.umap(adata, color=["leiden_res_0.02", "leiden_res_0.50", "leiden_res_2.00"], legend_loc="on data")

    # Construct the output path with the specified output directory and suffix "_processed.h5ad"
    output_path = os.path.join(output_dir, os.path.basename(h5ad_file).replace(".h5ad", "_processed.h5ad"))
    
    # Save the processed AnnData object for future analysis
    adata.write(output_path)
    print(f"Saved processed data to {output_path}\n")

In [4]:
import scanpy as sc

# Load the .h5ad file
adata = sc.read('Modified-h5ad/Trachea_TSP1_30_version2d_10X_smartseq_scvi_Nov122024_modified.h5ad')

donor_filtered_data = adata[(adata.obs['donor'] == 'TSP27') & (adata.obs['method'] == '10X')]
# Count the number of cells (rows in adata.obs or adata.X)
num_cells = donor_filtered_data.n_obs
print(f"Number of cells: {num_cells}")

Number of cells: 246
